# Activity 3: DataFrame Fundamentals (pandas and Polars)

**Module:** Week 2 Day 3
**Estimated time:** 60 to 80 minutes
**Type:** Guided notebook
**Libraries:** `pandas`, `polars`
**AI-free zone:** write the code yourself.

## What you will learn

- How to turn a list of dicts (exactly what an API call gives you) into a
  DataFrame
- The three moves you make on almost every new dataset: `.head()`, `.info()`,
  `.describe()`
- How to select columns, filter rows, and add a derived column in pandas
  3.0+'s recommended style
- The same four moves in Polars, and how its syntax differs from pandas
- Why real-world CSVs need cleaning even when an API's JSON usually does not

## The bridge from Activity 1

In Activity 1 you called Open-Meteo for 8 cities and built `weather_records`,
a list of dicts. That is exactly the shape a DataFrame is built from. If you
still have that list in your Activity 1 notebook, copy it into the cell below.
If you do not (key still activating, or you lost the notebook), run the
fallback cell instead so you can keep moving.

## Setup

Activity 0 created the shared environment at the repo root and you selected its
`.venv` as this notebook's kernel, so `pandas` and `polars` are already
installed. If they are missing, add them from the repo root:

```bash
uv add pandas polars
```

In [ ]:
# Paste your real weather_records list from Activity 1 here if you have it.
# If you do not have it yet, leave this as None and run the fallback cell below.
weather_records = None

In [ ]:
# Fallback: only runs if you did not paste real data above.
if weather_records is None:
    weather_records = [
        {"city": "Hartford,US", "temp_f": 71.6, "humidity": 58, "conditions": "broken clouds"},
        {"city": "New York,US", "temp_f": 74.1, "humidity": 52, "conditions": "clear sky"},
        {"city": "Boston,US", "temp_f": 68.9, "humidity": 61, "conditions": "overcast clouds"},
        {"city": "Chicago,US", "temp_f": 79.5, "humidity": 45, "conditions": "clear sky"},
        {"city": "Denver,US", "temp_f": 84.2, "humidity": 22, "conditions": "clear sky"},
        {"city": "Seattle,US", "temp_f": 65.3, "humidity": 70, "conditions": "light rain"},
        {"city": "Miami,US", "temp_f": 88.7, "humidity": 75, "conditions": "scattered clouds"},
        {"city": "Phoenix,US", "temp_f": 98.1, "humidity": 15, "conditions": "clear sky"},
    ]
    print("using fallback data (8 cities)")
else:
    print(f"using your real data ({len(weather_records)} cities)")

## 1. From a list of dicts to a DataFrame

This is the single most common way a DataFrame gets created in a real
pipeline: you already have Python objects (from an API, a database query, or
a file you parsed by hand), and you hand them to `pd.DataFrame(...)` in one
line.

In [ ]:
import pandas as pd

df = pd.DataFrame(weather_records)
df

## 2. The three moves you make on every new dataset

Before you clean or analyze anything, look at what you actually have.

In [ ]:
print(df.head(3))     # first few rows: does the data look like what you expected?
print()
print(df.info())      # column names, dtypes, non-null counts: any surprises?
print()
print(df.describe())  # summary stats for numeric columns only

**Expected observations**: `.head()` shows real rows so you can sanity-check
the shape. `.info()` shows `temp_f` and `humidity` as numeric (`float64`/
`int64`) and `city`/`conditions` as `str` (pandas 3.0's default string dtype),
with 8 non-null values in every column since this data came from a clean API. `.describe()` only
summarizes the numeric columns: count, mean, std, min, quartiles, max.

**PAUSE**: if one city's API call had silently failed and you appended
`None` for its temperature instead of skipping it, how would `.info()` show
that? (Answer: a non-null count lower than 8 for that column.)

## 3. Selecting columns and filtering rows

Selecting a column returns a `Series`; selecting a list of columns returns a
DataFrame. Filtering uses a boolean mask: an expression that is `True`/`False`
for every row, passed inside `[]`.

In [ ]:
# One column (a Series)
print(df["temp_f"])

# Multiple columns (still a DataFrame)
print(df[["city", "temp_f"]])

# Filter: only cities warmer than 80F
warm = df[df["temp_f"] > 80]
print(warm)

## 4. Adding a derived column, the pandas 3.0+ way

pandas 3.0 defaults to copy-on-write, which changes how you should write
assignments. **Use `.loc[row_mask, "column"] = value` or a plain new-column
assignment on the DataFrame you intend to modify.** Do not chain two selections
together and assign to the result (`df[df["x"] > 1]["y"] = 2`), because that
chain may operate on a temporary copy and silently not update `df` at all.

In [ ]:
# A new column computed for every row: safe, direct assignment
df["temp_c"] = ((df["temp_f"] - 32) * 5 / 9).round(1)

# A conditional update to an existing column: use .loc, not chained indexing
df.loc[df["temp_f"] >= 85, "heat_alert"] = True
df["heat_alert"] = df["heat_alert"].fillna(False)

print(df[["city", "temp_f", "temp_c", "heat_alert"]])

**Expected observation**: `heat_alert` is `True` only for cities at or above
85F (Miami and Phoenix in the fallback data). `df[df["x"] > 1]["y"] = 2` (two
`[]` in a row, "chained indexing") is the single most common pandas mistake
beginners make; `.loc[row_mask, "col"] = value` (one `[]`, both parts inside
it) is the fix, and it is required, not optional, in pandas 3.0+.

## 5. Sorting and a simple aggregate

You do not need `groupby` for everything. Sorting and a plain `.mean()` answer
plenty of real questions on their own.

In [ ]:
hottest_first = df.sort_values("temp_f", ascending=False)
print(hottest_first[["city", "temp_f"]])

print(f"\naverage temperature across all cities: {df['temp_f'].mean():.1f}F")
print(f"average humidity: {df['humidity'].mean():.1f}%")

## 6. The same moves in Polars

Polars is not a drop-in replacement for pandas: it has its own API, built
around **expressions** rather than in-place mutation. The shape of the work
is identical (create, inspect, filter, derive, sort), but the syntax differs
in a few specific, learnable ways.

In [ ]:
import polars as pl

pldf = pl.DataFrame(weather_records)
print(pldf.head(3))
print()
print(pldf.schema)     # Polars' equivalent of dtypes
print()
print(pldf.describe())

**Key difference #1**: Polars has no index. Every row is just a row; there is
no separate index column to align on, which removes a whole category of
pandas bugs (misaligned indexes after a filter or a merge).

In [ ]:
# Filtering: .filter() with a Polars expression, not [] with a boolean Series
warm_pl = pldf.filter(pl.col("temp_f") > 80)
print(warm_pl)

# Deriving columns: .with_columns(), not a bare assignment
pldf = pldf.with_columns([
    ((pl.col("temp_f") - 32) * 5 / 9).round(1).alias("temp_c"),
    (pl.col("temp_f") >= 85).alias("heat_alert"),
])
print(pldf.select(["city", "temp_f", "temp_c", "heat_alert"]))

**Key difference #2**: `pl.col("name")` builds an **expression**, a
description of a computation, not an immediate value. `.with_columns()` takes
a list of expressions and evaluates all of them at once. This is also why
Polars can be fast: it can look at every expression together and decide the
most efficient way to run them, instead of doing one Python-level operation
at a time the way pandas does.

In [ ]:
hottest_first_pl = pldf.sort("temp_f", descending=True)
print(hottest_first_pl.select(["city", "temp_f"]))

print(f"\naverage temperature: {pldf['temp_f'].mean():.1f}F")

## pandas vs Polars, at a glance

| Task | pandas | Polars |
|---|---|---|
| Create from list of dicts | `pd.DataFrame(records)` | `pl.DataFrame(records)` |
| First rows | `.head()` | `.head()` |
| Column types | `.info()` / `.dtypes` | `.schema` |
| Filter rows | `df[df["x"] > 1]` | `df.filter(pl.col("x") > 1)` |
| New column | `df["y"] = ...` | `df.with_columns(...)` |
| Row index | Yes, always present | No index |
| Execution model | Eager, one operation at a time | Eager by default, expressions can be optimized together; a separate lazy mode exists for large data |

You will go deeper on Polars, including its lazy mode and a real
pandas-vs-Polars timing comparison on a much bigger dataset, on Day 4.

## 7. A quick look at messier data

APIs like Open-Meteo usually hand you clean, well-typed JSON. Files that come
from other systems, exports, spreadsheets, legacy databases, often do not.
Here is a small, real, messy CSV (movie box office data) to see the
difference before Day 4 goes deeper on cleaning.

In [ ]:
messy = pd.DataFrame({
    "movie": ["Film A", "Film B", "Film C", "Film D"],
    "daily_gross": ["$1,204,500", "$980,300", "N/A", "$2,150,000"],
    "theaters": [3200, 2900, 2900, "3,500"],
})
print(messy)
print()
print(messy.dtypes)  # notice daily_gross and theaters are "object" (text), not numbers!

**Expected observation**: `daily_gross` and `theaters` show up as `object`
dtype, meaning pandas is treating them as text, because of the `$`, commas,
and the `"N/A"` string mixed in with numbers. You cannot `.mean()` a column of
text. Cleaning steps like this, strip formatting characters, convert types,
decide what to do with `"N/A"`, are most of what Day 4 covers in depth.

## Checkpoint

Show a neighbor: your `df` with `temp_c` and `heat_alert` columns, the same
result in Polars, and one sentence on why `messy["daily_gross"]` cannot be
averaged yet. Save this notebook; Day 4 picks up exactly where this leaves
off.